___

<a href='http://www.pieriandata.com'><img src='../Pierian_Data_Logo.png'/></a>
___
<center><em>Авторские права принадлежат Pierian Data Inc.</em></center>
<center><em>Для дополнительной информации посетите наш сайт <a href='http://www.pieriandata.com'>www.pieriandata.com</a></em></center>

# Проверочный проект по линейной регрессии

Мы изучили построение признаков, кросс-валидацию и поиск по сетке. Пришло время проверить Ваши новые навыки. Добро пожаловать на проверочный проект по машинному обучению! В этом проекте у нас будут достаточно подробные пошаговые инструкции. Будущие проекты будут уже в более свободной форме. Мы начнём проект с финальной версии набора данных Ames Housing, с которым мы работали в разделе "построение признаков" этого курса. Цель этого проекта - создать модель линейной регрессии, обучить её на данных с поиском оптимальных значений параметров по сетке, и затем оценить модель на тестовом наборе данных.

---
---
---
## Выполните задания, написанные жирным шрифтом

**ЗАДАНИЕ: Выполните ячейки ниже для импорта библиотек и загрузки данных. Возможно в будущем Вам понадобятся дополнительные команды import от scikit-learn.**

### Imports

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

### Данные

In [5]:
df = pd.read_csv("../DATA/AMES_Final_DF.csv")

FileNotFoundError: [Errno 2] No such file or directory: '../DATA/AMES_Final_DF.txt'

In [6]:
df.head()

,Unnamed: 0,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,...,Sale Type_ConLw,Sale Type_New,Sale Type_Oth,Sale Type_VWD,Sale Type_WD,Sale Condition_AdjLand,Sale Condition_Alloca,Sale Condition_Family,Sale Condition_Normal,Sale Condition_Partial
0,0,141.0,31770,6,5,1960,1960,112.0,639.0,0.0,...,False,False,False,False,True,False,False,False,True,False
1,1,80.0,11622,5,6,1961,1961,0.0,468.0,144.0,...,False,False,False,False,True,False,False,False,True,False
2,2,81.0,14267,6,6,1958,1958,108.0,923.0,0.0,...,False,False,False,False,True,False,False,False,True,False
3,3,93.0,11160,7,5,1968,1968,0.0,1065.0,0.0,...,False,False,False,False,True,False,False,False,True,False
4,4,74.0,13830,5,5,1997,1998,0.0,791.0,0.0,...,False,False,False,False,True,False,False,False,True,False


In [69]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Columns: 264 entries, Unnamed: 0 to Sale Condition_Partial
dtypes: bool(227), float64(11), int64(26)
memory usage: 1.5 MB


In [7]:
df_two = df[list(df.nunique()[df.nunique() > 2].index)].drop('Unnamed: 0', axis = 1)
df_two.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2925 entries, 0 to 2924
Data columns (total 36 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Lot Frontage     2925 non-null   float64
 1   Lot Area         2925 non-null   int64  
 2   Overall Qual     2925 non-null   int64  
 3   Overall Cond     2925 non-null   int64  
 4   Year Built       2925 non-null   int64  
 5   Year Remod/Add   2925 non-null   int64  
 6   Mas Vnr Area     2925 non-null   float64
 7   BsmtFin SF 1     2925 non-null   float64
 8   BsmtFin SF 2     2925 non-null   float64
 9   Bsmt Unf SF      2925 non-null   float64
 10  Total Bsmt SF    2925 non-null   float64
 11  1st Flr SF       2925 non-null   int64  
 12  2nd Flr SF       2925 non-null   int64  
 13  Low Qual Fin SF  2925 non-null   int64  
 14  Gr Liv Area      2925 non-null   int64  
 15  Bsmt Full Bath   2925 non-null   float64
 16  Bsmt Half Bath   2925 non-null   float64
 17  Full Bath     

In [8]:
nulls = df.isnull().sum().sort_values(axis = 0)
nulls
# нет nan-ов в датафрейме
df_two_int = df_two.select_dtypes(include= ['float', 'int64'])
def clean_outfliers(series):
    q1, q3 = np.percentile(series, q = [25, 75])
    IQR = (q3 - q1) * 1.5
    return (series > q1 - IQR) & (series < q3 + IQR)

# Пишем маску, при которой в рамках одного дома количество выбросов меньше 36 - 24
mask = df_two_int.apply(clean_outfliers)
df_two = df_two_int[mask.sum(axis = 1) > 24]
df_two

# Не float и int столбцов, поэтому можем отсавить не объединяя с exclude = ['int64', 'float']

,Lot Frontage,Lot Area,Overall Qual,Overall Cond,Year Built,Year Remod/Add,Mas Vnr Area,BsmtFin SF 1,BsmtFin SF 2,Bsmt Unf SF,...,Wood Deck SF,Open Porch SF,Enclosed Porch,3Ssn Porch,Screen Porch,Pool Area,Misc Val,Mo Sold,Yr Sold,SalePrice
0,141.000000,31770,6,5,1960,1960,112.0,639.0,0.0,441.0,...,210,62,0,0,0,0,0,5,2010,215000
1,80.000000,11622,5,6,1961,1961,0.0,468.0,144.0,270.0,...,140,0,0,0,120,0,0,6,2010,105000
2,81.000000,14267,6,6,1958,1958,108.0,923.0,0.0,406.0,...,393,36,0,0,0,0,12500,6,2010,172000
3,93.000000,11160,7,5,1968,1968,0.0,1065.0,0.0,1045.0,...,0,0,0,0,0,0,0,4,2010,244000
4,74.000000,13830,5,5,1997,1998,0.0,791.0,0.0,137.0,...,212,34,0,0,0,0,0,3,2010,189900
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2920,37.000000,7937,6,6,1984,1984,0.0,819.0,0.0,184.0,...,120,0,0,0,0,0,0,3,2006,142500
2921,75.144444,8885,5,5,1983,1983,0.0,301.0,324.0,239.0,...,164,0,0,0,0,0,0,6,2006,131000
2922,62.000000,10441,5,5,1992,1992,0.0,337.0,0.0,575.0,...,80,32,0,0,0,0,700,7,2006,132000
2923,77.000000,10010,5,5,1974,1975,0.0,1071.0,123.0,195.0,...,240,38,0,0,0,0,0,4,2006,170000


**ЗАДАНИЕ: Мы будем пытаться спрогнозировать значение колонки SalePrice. Разделите данные на две части - признаки X и целевая переменная y.**

In [21]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.linear_model import ElasticNet
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, get_scorer_names

**ЗАДАНИЕ: С помощью scikit-learn разделите и X, и y на обучающий и тестовый наборы данных. Поскольку далее мы будем использовать поиск по сетке, то выделите под тестовые данные 10% от всех данных. Чтобы получить такое же разбиение данных, как и в нашем блокноте, можете использовать random_state = 101.**

In [22]:
X = df_two.drop('SalePrice', axis = 1)
y = df_two['SalePrice']


X_train, X_test, y_train, y_test = train_test_split(X, y, test_size= 0.1, random_state= 101)


**ЗАДАНИЕ: Признаки в этом наборе данных имеют различный масштаб и различные единицы измерения. Для оптимальной работы регрессии, выполните масштабирование признаков X. Обратите внимание на то, какие данные подавать на вход для .fit(), а какие данные на вход для .transform().**

In [23]:
standartScaler = StandardScaler()
standartScaler.fit(X_train)
X_train = standartScaler.transform(X_train)
X_test = standartScaler.transform(X_test)
X_train.shape

(2464, 35)

**ЗАДАНИЕ: Мы будем использовать модель "Elastic Net". Создайте экземпляр модели ElasticNet в scikit-learn, используя параметры по умолчанию.**

In [24]:
elasticnet = ElasticNet()


**ЗАДАНИЕ: Модель Elastic Net работает с двумя основными параметрами - alpha и L1_ratio. Создайте словарь с набором различных значений этих параметров, для поиска по сетке. Вы можете выбрать различные значения параметров, но имейте ввиду, что Ваши результаты могут не совпасть с результатами в блокноте с решениями.**

In [25]:
grid = {'alpha': [1., 3., 100], 'l1_ratio': [0.1, 0.5, 1]}

**ЗАДАНИЕ: С помощью scikit-learn создайте объект GridSearchCV и запустите поиск по сетке для нахождения наилучших параметров модели, используя обучающие данные (предварительно смасштабированные). [Для некоторых комбинаций параметров Вы можете получить предупреждения (warnings).](https://stackoverflow.com/questions/20681864/lasso-on-sklearn-does-not-converge)**

In [27]:
gridsearchCV = GridSearchCV(elasticnet, param_grid= grid, cv = 5, scoring= 'neg_mean_squared_error')

print(get_scorer_names())
gridsearchCV.fit(X_train, y_train)
gridsearchCV.best_params_

print('====Лучшая модель====')
print(gridsearchCV.best_params_, gridsearchCV.best_estimator_.coef_, gridsearchCV.best_score_)

['accuracy', 'adjusted_mutual_info_score', 'adjusted_rand_score', 'average_precision', 'balanced_accuracy', 'completeness_score', 'd2_absolute_error_score', 'explained_variance', 'f1', 'f1_macro', 'f1_micro', 'f1_samples', 'f1_weighted', 'fowlkes_mallows_score', 'homogeneity_score', 'jaccard', 'jaccard_macro', 'jaccard_micro', 'jaccard_samples', 'jaccard_weighted', 'matthews_corrcoef', 'mutual_info_score', 'neg_brier_score', 'neg_log_loss', 'neg_max_error', 'neg_mean_absolute_error', 'neg_mean_absolute_percentage_error', 'neg_mean_gamma_deviance', 'neg_mean_poisson_deviance', 'neg_mean_squared_error', 'neg_mean_squared_log_error', 'neg_median_absolute_error', 'neg_negative_likelihood_ratio', 'neg_root_mean_squared_error', 'neg_root_mean_squared_log_error', 'normalized_mutual_info_score', 'positive_likelihood_ratio', 'precision', 'precision_macro', 'precision_micro', 'precision_samples', 'precision_weighted', 'r2', 'rand_score', 'recall', 'recall_macro', 'recall_micro', 'recall_samples'

**ЗАДАНИЕ: Отобразите наилучшую комбинацию параметров для Вашей модели.**

In [28]:
gridsearchCV.best_params_

{'alpha': 100, 'l1_ratio': 1}

**ЗАДАНИЕ: Оцените работу модели на тестовом наборе данных (предварительно смасштабированном) в 10%, которые модель ещё не видела. В блокноте с решениями мы получили MAE = $\$$14149 и RMSE = $\$$20532**

In [29]:
y_predict = gridsearchCV.predict(X_test)
MAE = mean_absolute_error(y_test, y_predict)
RMSE = np.sqrt(mean_squared_error(y_test, y_predict))
r2 = r2_score(y_test, y_predict)
print(MAE, RMSE, r2, 'по r2 видно, что можель предсказывает удачно')

16626.94446539061 23881.62876363722 0.8650174670663351 по r2 видно, что можель предсказывает удачно


14149.055026374837

20532.890234901013

180815.53743589742

## Отличная работа!

----

# ПОЛИНОМИАЛЬНАЯ регрессия

In [54]:
from sklearn.preprocessing import PolynomialFeatures
from sklearn.feature_selection import SelectKBest, f_regression

X_train_poly = X_train
polyConverter = PolynomialFeatures(degree= 2, include_bias= True)
print(X_train_poly.shape)
X_train_poly = polyConverter.fit_transform(X_train_poly)
X_test_poly = polyConverter.transform(X_test)
print(X_train_poly.shape, ' - shape after polynomial transform')

selector_features = SelectKBest(k= 100, score_func=f_regression)
selector_features.fit(X_train_poly, y_train)
X_train_poly = selector_features.transform(X_train_poly)
X_test_poly = selector_features.transform(X_test_poly)
print(X_train_poly.shape, ' - after selection 100 features')

gridsearchCV.fit(X_train, y_train)
print('===estimator===')
print(gridsearchCV.best_params_, gridsearchCV.best_estimator_.coef_)

(2464, 35)
(2464, 666)  - shape after polynomial transform
(2464, 100)  - after selection 100 features
===estimator===
{'alpha': 10, 'l1_ratio': 1} [ 3112.25690181  3173.6276199  18636.20609446  4555.47904676
 10430.5877334   4682.74716195  1830.97816898  7721.17253037
   887.06759052    -0.          6483.70907307   442.75885087
     0.         -1208.83740606 21741.16947052   984.74682435
  -469.11675503     0.         -1306.83699092 -5484.62127146
 -4389.83735843  3392.66083633  2183.3565506  -3633.72869147
  3609.19849177  4620.21393005  1215.59417456   625.0240382
  1113.82700383   124.20404431  1288.12156361   381.95431098
   -55.11853498   896.62513891  -912.94550307]


In [55]:
grid = {'alpha': [1., 0.1, 3., 10], 'l1_ratio': [0.1, 0.5, 1]}
elasticnet = ElasticNet(max_iter=10000, tol=1e-4, random_state=42)
gridsearchCV_poly = GridSearchCV(elasticnet, param_grid= grid, cv = 3, scoring= 'neg_mean_squared_error')
gridsearchCV_poly.fit(X_train_poly, y_train)

y_predict_for_poly = gridsearchCV_poly.predict(X_test_poly)
MAE = mean_absolute_error(y_predict_for_poly, y_test)
RMSE = np.sqrt(mean_squared_error(y_test, y_predict_for_poly))
print(MAE, RMSE, 'Что ниже метрик качества на линейной регрессии - гарантирует меньше потерь и больше точности')

/home/abu/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.403e+11, tolerance: 6.532e+08
  model = cd_fast.enet_coordinate_descent(
/home/abu/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.310e+11, tolerance: 6.529e+08
  model = cd_fast.enet_coordinate_descent(
/home/abu/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2

14716.009519721316 21889.956649546977


/home/abu/anaconda3/lib/python3.13/site-packages/sklearn/linear_model/_coordinate_descent.py:695: ConvergenceWarning: Objective did not converge. You might want to increase the number of iterations, check the scale of the features or consider increasing regularisation. Duality gap: 2.504e+10, tolerance: 6.623e+08
  model = cd_fast.enet_coordinate_descent(
